In [ ]:
import pandas as pd
df = pd.read_csv('/content/Project2_Customer_Retention_50000_Rows_8_Columns.csv')

In [ ]:
#inspecting the data types
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          50000 non-null  object 
 1   SignupDate          50000 non-null  object 
 2   Region              50000 non-null  object 
 3   AcquisitionChannel  50000 non-null  object 
 4   PlanName            50000 non-null  object 
 5   MonthlyFee          50000 non-null  float64
 6   Status              50000 non-null  object 
 7   SupportTicketCount  50000 non-null  int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 3.1+ MB
None
CustomerID            0
SignupDate            0
Region                0
AcquisitionChannel    0
PlanName              0
MonthlyFee            0
Status                0
SupportTicketCount    0
dtype: int64


In [ ]:
#Convert SignupDate to datetime format
df['SignupDate'] = pd.to_datetime(df['SignupDate'])

In [ ]:
#Define an observation date to calculate tenure
observation_date = pd.to_datetime('2025-12-31')
df['TenureDays'] = (observation_date - df['SignupDate']).dt.days
df['TenureMonths'] = (df['TenureDays'] / 30.44).round(1)

In [ ]:
#Create a numeric Churn Flag
df['ChurnFlag'] = df['Status'].apply(lambda x: 1 if x == 'Churned' else 0)

In [ ]:
print("Data inspection and feature engineering complete!")
print(df.head(2))

Data inspection and feature engineering complete!
  CustomerID SignupDate Region AcquisitionChannel         PlanName  \
0     C06578 2023-07-06   West     Email Campaign  Premium Monthly   
1     C03785 2023-08-28  North     Email Campaign  Premium Monthly   

   MonthlyFee   Status  SupportTicketCount  TenureDays  TenureMonths  \
0       34.99   Active                   1         909          29.9   
1       31.49  Churned                   1         856          28.1   

   ChurnFlag  
0          0  
1          1  


In [ ]:
#SQLlite Analysis

In [ ]:
import sqlite3

# Connect to local SQLite database
conn = sqlite3.connect('customer_retention_50k.db')
cursor = conn.cursor()

# Load dataframe into SQLite
df.to_sql('Customers', conn, if_exists='replace', index=False)

# SQL Query: Churn Rate, Total Revenue Lost, and Avg Support Tickets by Plan
query = """
SELECT
    PlanName,
    COUNT(CustomerID) AS TotalCustomers,
    SUM(ChurnFlag) AS ChurnedCustomers,
    ROUND(CAST(SUM(ChurnFlag) AS FLOAT) / COUNT(CustomerID) * 100, 2) AS ChurnRatePercent,
    ROUND(AVG(SupportTicketCount), 2) AS AvgSupportTickets,
    ROUND(SUM(CASE WHEN Status = 'Churned' THEN MonthlyFee ELSE 0 END), 2) AS MonthlyRevenueLost
FROM Customers
GROUP BY PlanName
ORDER BY ChurnRatePercent DESC;
"""

df_analysis = pd.read_sql(query, conn)
print(df_analysis)

conn.close()

           PlanName  TotalCustomers  ChurnedCustomers  ChurnRatePercent  \
0     Basic Monthly           12398              3523             28.42   
1  Standard Monthly           15024              2999             19.96   
2   Premium Monthly            9073              1360             14.99   
3   Standard Annual            8486               853             10.05   
4    Premium Annual            5019               432              8.61   

   AvgSupportTickets  MonthlyRevenueLost  
0               1.37            34324.27  
1               1.24            58642.01  
2               1.20            46396.40  
3               1.12            13831.94  
4               1.12            12242.81  


In [ ]:
import os
import pandas as pd
from google.colab import files

# 1. Save the dataframe locally in the Colab cloud environment
file_path = 'Project2_Customer_Retention_Processed.csv'
df.to_csv(file_path, index=False)
print(f"File successfully saved locally to: {file_path}")

# 2. Trigger a browser download to your computer's local Downloads folder
files.download(file_path)

File successfully saved locally to: Project2_Customer_Retention_Processed.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>